# Part 5A: Continuous Batching Foundations

So far, our inference engine can maintain multiple requests using states such as:

```text
waiting
running
finished
```

We also built a scheduler that decides which waiting requests are allowed to run.

However, there is still an important limitation.

## The problem with fixed batches

Suppose our engine can run two requests at once:

```text
Running capacity = 2

Request A → needs 3 generated tokens
Request B → needs 6 generated tokens
Request C → waiting
```

If we create a fixed batch containing `A` and `B`, decoding may look like this:

```text
Iteration 1: A token 1 | B token 1
Iteration 2: A token 2 | B token 2
Iteration 3: A token 3 | B token 3   ← A finishes

Iteration 4:     empty | B token 4
Iteration 5:     empty | B token 5
Iteration 6:     empty | B token 6   ← B finishes
```

Even though request `A` finished after iteration 3, request `C` remains waiting until the entire batch finishes.

That leaves available execution capacity unused.

For an online LLM serving system, requests can arrive and finish at completely different times, so treating a batch as fixed for its entire lifetime is inefficient.

---

## Continuous batching

Continuous batching changes **when scheduling happens**.

Instead of doing:

```text
schedule requests
      ↓
run the entire batch until every request finishes
      ↓
schedule again
```

the engine repeatedly performs:

```text
schedule
   ↓
execute one model iteration
   ↓
update request states
   ↓
schedule again
   ↓
execute another iteration
   ↓
...
```

The scheduler therefore gets another opportunity to choose work between model iterations.

Using the same example:

```text
Iteration 1:
[A, B]

Iteration 2:
[A, B]

Iteration 3:
[A, B]

A finishes
```

Before the next iteration, the scheduler sees that one execution slot is now available and admits request `C`.

```text
Iteration 4:
[C, B]

Iteration 5:
[C, B]
```

The active batch can therefore change while generation is happening.

For example:

```text
Iteration 1   [A, B]

Iteration 2   [A, B]

Iteration 3   [A, B]
                 ↓
              A finishes

Iteration 4   [C, B]

Iteration 5   [C, D]

Iteration 6   [E, D]
```

Requests can enter and leave the active set between iterations.

This is the central idea behind **continuous batching**, also commonly called **iteration-level scheduling**.

---

## What is an iteration?

During autoregressive decoding, one decoding iteration roughly performs:

```text
current token
     ↓
model forward pass
     ↓
logits
     ↓
select next token
     ↓
update KV cache
```

For a decoding request, one iteration usually advances that request by one generated token.

The serving loop can therefore be thought of roughly as:

```python
while engine_has_work:

    schedule_requests()

    execute_model()

    sample_tokens()

    update_requests()
```

After each iteration, some requests may still be running while others may have finished.

```text
running request
      │
      ▼
model iteration
      │
      ▼
new token
      │
      ├── not finished ──→ remain running
      │
      └── finished ──────→ move to finished
```

Once a request finishes, its execution capacity can immediately be reused by another waiting request during the next scheduling iteration.

---

## Why is it called continuous batching?

The GPU is not literally executing one endless batch.

The word **continuous** refers to the fact that requests can continuously enter and leave the active batch between model iterations.

Traditional batching behaves more like:

```text
Batch 1: [A, B]

wait until both finish

Batch 2: [C, D]

wait until both finish
```

Continuous batching behaves more like:

```text
[A, B]
[A, B]
[A, B]
[C, B]
[C, D]
[E, D]
...
```

So the important mental model is:

> A batch is no longer a fixed collection of requests. It is a continuously changing set of active sequences chosen by the scheduler at iteration boundaries.

---

## What we will build

We will extend the scheduler from the previous part so that our engine can:

* execute requests one decoding iteration at a time
* repeatedly schedule work between iterations
* detect when individual requests finish
* remove finished requests immediately
* admit waiting requests into newly available capacity
* maintain a changing set of active sequences
* execute multiple active requests together
* handle requests with different sequence lengths
* introduce a token-based scheduling budget
* understand how prefill work and decode work interact

By the end of this part, our engine will no longer think in terms of:

```text
create batch → finish entire batch → create next batch
```

Instead, it will operate as a continuous serving loop:

```text
schedule
   ↓
execute
   ↓
update
   ↓
schedule
   ↓
execute
   ↓
...
```

This is the foundation we need before moving into KV-cache management and paged allocation.


In [1]:
#initial setup

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache

# Model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16
).cuda()

model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [2]:
waiting = []
running = []
finished = []

max_running = 2

In [3]:
def create_request(request_id, max_new_tokens):
    return {
        "id": request_id,
        "max_new_tokens": max_new_tokens,
        "generated_tokens": 0,
        "finished": False,
    }

In [4]:
waiting.append(create_request("A", 3))
waiting.append(create_request("B", 6))
waiting.append(create_request("C", 4))

In [5]:
def schedule():
    while waiting and len(running) < max_running:
        request = waiting.pop(0)
        running.append(request)

In [6]:
def execute_iteration():
    for request in running:
        request["generated_tokens"] += 1

In [7]:
def check_finished():
    for request in running:
        if request["generated_tokens"] >= request["max_new_tokens"]:
            request["finished"] = True

In [8]:
def remove_finished():
    for request in running[:]:
        if request["finished"]:
            running.remove(request)
            finished.append(request)

In [9]:
while waiting or running:

    schedule()

    execute_iteration()

    check_finished()

    remove_finished()

In [11]:
print(waiting,running,finished )

[] [] [{'id': 'A', 'max_new_tokens': 3, 'generated_tokens': 3, 'finished': True}, {'id': 'B', 'max_new_tokens': 6, 'generated_tokens': 6, 'finished': True}, {'id': 'C', 'max_new_tokens': 4, 'generated_tokens': 4, 'finished': True}]


## Building the First Continuous Batching Loop

We will first simulate generation without calling the model.

The goal is to isolate the scheduling behavior:

```text
schedule
   ↓
execute one iteration
   ↓
check for completed requests
   ↓
remove them
   ↓
repeat
```

### Request state

```python
waiting = []
running = []
finished = []

max_running = 2
```

Each request stores how many tokens it should generate and how many it has generated so far.

```python
def create_request(request_id, max_new_tokens):
    return {
        "id": request_id,
        "max_new_tokens": max_new_tokens,
        "generated_tokens": 0,
        "finished": False,
    }
```

Add three requests:

```python
waiting.append(create_request("A", 3))
waiting.append(create_request("B", 6))
waiting.append(create_request("C", 4))
```

Initially:

```text
waiting  = [A, B, C]
running  = []
finished = []
```

---

### 1. Schedule requests

The scheduler moves waiting requests into `running` while execution capacity is available.

```python
def schedule():
    while waiting and len(running) < max_running:
        request = waiting.pop(0)
        running.append(request)
```

The condition:

```python
while waiting
```

means:

```python
while len(waiting) > 0
```

A non-empty Python list evaluates to `True`, while an empty list evaluates to `False`.

With `max_running = 2`, the first scheduling step produces:

```text
waiting = [C]
running = [A, B]
```

---

### 2. Execute one iteration

For now, we simulate one decoding iteration by advancing every running request by one token.

```python
def execute_iteration():
    for request in running:
        request["generated_tokens"] += 1
```

For example:

```text
Before:
A = 0 / 3
B = 0 / 6

After one iteration:
A = 1 / 3
B = 1 / 6
```

The important point is that we only advance requests **one iteration at a time** rather than completing them immediately.

---

### 3. Check for finished requests

After every iteration, we determine whether any request has generated all of its allowed tokens.

```python
def check_finished():
    for request in running:
        if request["generated_tokens"] >= request["max_new_tokens"]:
            request["finished"] = True
```

For example:

```text
A = 3 / 3 → finished
B = 3 / 6 → still running
```

---

### 4. Remove finished requests

Finished requests are moved out of `running`.

```python
def remove_finished():
    for request in running[:]:
        if request["finished"]:
            running.remove(request)
            finished.append(request)
```

We iterate over:

```python
running[:]
```

which creates a shallow copy of the list.

This allows us to safely remove elements from the original `running` list while iterating.

---

## Continuous batching loop

Now we combine the four operations:

```python
while waiting or running:

    schedule()

    execute_iteration()

    check_finished()

    remove_finished()
```

The outer `while` loop is what repeatedly returns execution to the scheduler.

Conceptually:

```text
schedule
   ↓
execute
   ↓
check
   ↓
remove
   ↓
        ┌───────────────┐
        │               │
        └── schedule ◀──┘
```

If a request finishes, the next iteration of the outer loop calls `schedule()` again, allowing another waiting request to immediately take the newly available slot.

---

### Final state

```python
print(waiting)
print(running)
print(finished)
```

Output:

```python
[]
[]

[
    {
        'id': 'A',
        'max_new_tokens': 3,
        'generated_tokens': 3,
        'finished': True
    },
    {
        'id': 'B',
        'max_new_tokens': 6,
        'generated_tokens': 6,
        'finished': True
    },
    {
        'id': 'C',
        'max_new_tokens': 4,
        'generated_tokens': 4,
        'finished': True
    }
]
```

At the end:

```text
waiting  = []
running  = []
finished = [A, B, C]
```

All requests have passed through the full lifecycle:

```text
waiting → running → finished
```

This gives us the basic control loop needed for continuous batching.

The next step is to replace the simulated:

```python
request["generated_tokens"] += 1
```

with actual model execution.


In [12]:
def create_request(request_id, prompt, max_new_tokens):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    return {
        "id": request_id,
        "prompt": prompt,

        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],

        "cache": DynamicCache(config=model.config),

        "generated_tokens": [],
        "max_new_tokens": max_new_tokens,

        "finished": False,
    }

In [13]:
waiting = []
running = []
finished = []

max_running = 2

In [14]:
waiting.append(
    create_request(
        "A",
        "The capital of France is",
        max_new_tokens=5
    )
)

waiting.append(
    create_request(
        "B",
        "The largest planet in our solar system is",
        max_new_tokens=8
    )
)

waiting.append(
    create_request(
        "C",
        "Machine learning is",
        max_new_tokens=6
    )
)

In [40]:
print("waiting: ",end="")
for i in waiting:
  print(i['id'],"\t",end="")
print("\nrunning: ",end="")
for j in running:
  print(j['id'],"\t",end="")

waiting: A 	B 	C 	
running: 

In [41]:
def schedule():
    while waiting and len(running) < max_running:
        request = waiting.pop(0)
        running.append(request)

In [42]:
@torch.inference_mode()
def execute_iteration():

    for request in running:

        outputs = model(
            input_ids=request["input_ids"],
            attention_mask=request["attention_mask"],
            past_key_values=request["cache"],
            use_cache=True,
        )

        # Save the updated KV cache
        request["cache"] = outputs.past_key_values

        # Logits shape:
        # [1, current_input_length, vocab_size]
        next_token = outputs.logits[:, -1, :].argmax(
            dim=-1,
            keepdim=True
        )

        # Store generated token ID
        request["generated_tokens"].append(
            next_token.item()
        )

        # During the next decode iteration,
        # only feed this new token.
        request["input_ids"] = next_token

        # Total context length increased by one token.
        request["attention_mask"] = torch.cat(
            [
                request["attention_mask"],
                torch.ones(
                    (1, 1),
                    dtype=request["attention_mask"].dtype,
                    device=model.device,
                ),
            ],
            dim=-1,
        )

In [43]:
def check_finished():

    for request in running:

        reached_max_tokens = (
            len(request["generated_tokens"])
            >= request["max_new_tokens"]
        )

        generated_eos = (
            len(request["generated_tokens"]) > 0
            and
            request["generated_tokens"][-1]
            == tokenizer.eos_token_id
        )

        if reached_max_tokens or generated_eos:
            request["finished"] = True

In [44]:
def remove_finished():

    for request in running[:]:

        if request["finished"]:
            running.remove(request)
            finished.append(request)

In [45]:
step = 0

while waiting or running:

    step += 1

    # Fill available execution slots
    schedule()

    print(
        f"Step {step} | Running:",
        [request["id"] for request in running]
    )

    # Actual model inference
    execute_iteration()

    # Check EOS / max token limit
    check_finished()

    # Free completed request slots
    remove_finished()

Step 1 | Running: ['A', 'B']
Step 2 | Running: ['A', 'B']
Step 3 | Running: ['A', 'B']
Step 4 | Running: ['A', 'B']
Step 5 | Running: ['A', 'B']
Step 6 | Running: ['B', 'C']
Step 7 | Running: ['B', 'C']
Step 8 | Running: ['B', 'C']
Step 9 | Running: ['C']
Step 10 | Running: ['C']
Step 11 | Running: ['C']


In [46]:
for request in finished:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True
    )

    print(f"\nRequest {request['id']}:")
    print(text)


Request A:
 Paris. It is the

Request B:
 ____
A. Venus
B.

Request C:
 a field of artificial intelligence that


## Adding Real Model Inference to the Continuous Scheduler

In the previous section, we proved that our scheduler could continuously move requests through:

```text
waiting → running → finished
```

However, generation itself was still simulated using:

```python
request["generated_tokens"] += 1
```

Now we will replace that fake token counter with **real autoregressive model inference**.

The scheduler will still operate one iteration at a time, but every running request will now:

1. process its prompt,
2. generate an actual token,
3. maintain its own KV cache,
4. reuse that cache during later decode iterations,
5. stop when it reaches EOS or `max_new_tokens`.

At this stage, we have **continuous scheduling**, but each request is still executed with its own model forward pass.

So if:

```text
running = [A, B]
```

we currently perform:

```text
model(A)
model(B)
```

rather than:

```text
model([A, B])
```

Actual batched GPU execution will be the next step.

---

# Creating a Real Inference Request

Previously, a request only needed enough state to simulate generation:

```python
{
    "id": "A",
    "generated_tokens": 0,
    "max_new_tokens": 5,
    "finished": False
}
```

Real model inference requires more information.

Each request must now carry its own:

```text
prompt
input tokens
attention mask
KV cache
generated token IDs
generation limit
finished state
```

We create that request using:

```python
def create_request(request_id, prompt, max_new_tokens):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    return {
        "id": request_id,
        "prompt": prompt,

        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],

        "cache": DynamicCache(config=model.config),

        "generated_tokens": [],
        "max_new_tokens": max_new_tokens,

        "finished": False,
    }
```

Let's break this down carefully.

---

## Tokenizing the Prompt

```python
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)
```

Suppose:

```python
prompt = "The capital of France is"
```

The tokenizer converts the text into token IDs.

Conceptually:

```text
"The capital of France is"

        ↓ tokenizer

[785, 6864, 315, 9625, 374]
```

The exact token IDs depend on the tokenizer.

Because we use:

```python
return_tensors="pt"
```

the tokenizer returns PyTorch tensors instead of Python lists.

So:

```python
inputs["input_ids"]
```

might have a shape such as:

```text
[1, 5]
```

The dimensions mean:

```text
[batch_size, sequence_length]
```

Since this request contains one prompt:

```text
batch_size = 1
```

and if the prompt contains five tokens:

```text
sequence_length = 5
```

---

## Moving Inputs to the GPU

We then use:

```python
.to(model.device)
```

If our model is loaded on CUDA, then:

```text
model.device = cuda
```

and both:

```python
input_ids
attention_mask
```

are moved onto the GPU.

Otherwise PyTorch would eventually complain because the model and its inputs must be on compatible devices.

---

# `input_ids`

We save:

```python
"input_ids": inputs["input_ids"]
```

At the beginning, this contains the entire prompt.

For example:

```text
shape = [1, 5]

[
    [785, 6864, 315, 9625, 374]
]
```

During the **first forward pass**, the model sees all five prompt tokens.

That first processing stage is the:

```text
PREFILL
```

After prefill, we will no longer send the entire prompt every time.

Instead, the KV cache stores information about the previous tokens.

---

# `attention_mask`

We also save:

```python
"attention_mask": inputs["attention_mask"]
```

For an unpadded five-token prompt, this might look like:

```text
[[1, 1, 1, 1, 1]]
```

with shape:

```text
[1, 5]
```

A `1` means:

```text
this token should participate in attention
```

A `0` is generally used for positions that should be ignored, such as padding.

For example, later when batching different prompt lengths, we may have:

```text
Request A:
[11, 22, 33, 44]

Request B:
[55, 66]
```

To place both into a rectangular tensor, B may need padding:

```text
A = [11, 22, 33, 44]
B = [ 0,  0, 55, 66]
```

The attention masks tell the model which positions are real:

```text
A mask = [1, 1, 1, 1]
B mask = [0, 0, 1, 1]
```

Without that information, the model could treat padding tokens as real context.

---

# Why Are We Using an Attention Mask Now?

This is worth separating from the code.

In our earlier single-request experiments, we had something much simpler:

```text
one request
no padding
one sequence
all positions valid
```

In that situation, an explicit attention mask was often unnecessary.

If every token in:

```python
input_ids
```

is valid, then conceptually the mask would simply be:

```text
[1, 1, 1, 1, 1]
```

There is nothing interesting for it to mask.

That allowed us to focus on the more important concepts at the time:

```text
prefill
decode
KV caching
manual autoregressive generation
```

Now our goal has changed.

We are building toward:

```text
multiple requests
different prompt lengths
different generated lengths
padding
batched execution
```

At that point, attention masks become important.

There is another reason related to KV caching.

Suppose the prompt contains five tokens.

After prefill:

```text
KV cache length = 5
```

Then we generate:

```text
" Paris"
```

During the next decode step, we only send the new token:

```text
input_ids shape = [1, 1]
```

But the model's effective context is not one token.

It is:

```text
5 cached tokens
+
1 current token
=
6 total tokens
```

Therefore the attention mask represents the complete context:

```text
[1, 1, 1, 1, 1, 1]
```

even though the current `input_ids` contains only one token.

This difference is important:

```text
input_ids
=
tokens being processed NOW

attention_mask
=
positions belonging to the complete visible context
```

Once caching and batching interact, keeping these separate becomes much more important.

---

# Creating a KV Cache for Each Request

Each request receives its own cache:

```python
"cache": DynamicCache(config=model.config)
```

Initially:

```text
cache = empty
```

When the prompt goes through the model:

```text
"The capital of France is"
             ↓
           MODEL
             ↓
       K/V states created
             ↓
          KV cache
```

The model no longer has to recompute those prompt tokens on every future decode step.

Each request needs its **own** cache because A, B, and C have completely different contexts.

Conceptually:

```text
A → KV cache A
B → KV cache B
C → KV cache C
```

We cannot share them.

---

# Generated Tokens

Instead of storing only a count:

```python
generated_tokens = 3
```

we now store the actual generated token IDs:

```python
"generated_tokens": []
```

For example, after three decode steps:

```python
request["generated_tokens"]
```

might contain:

```text
[12095, 13, 1084]
```

Later:

```python
tokenizer.decode(...)
```

converts those IDs back into text.

---

# Creating Our Requests

We reset our three lifecycle queues:

```python
waiting = []

running = []

finished = []

max_running = 2
```

Then create three actual inference requests:

```python
waiting.append(
    create_request(
        "A",
        "The capital of France is",
        max_new_tokens=5
    )
)

waiting.append(
    create_request(
        "B",
        "The largest planet in our solar system is",
        max_new_tokens=8
    )
)

waiting.append(
    create_request(
        "C",
        "Machine learning is",
        max_new_tokens=6
    )
)
```

So initially:

```text
waiting  = [A, B, C]
running  = []
finished = []
```

Each request already contains its:

```text
tokenized prompt
attention mask
empty KV cache
generation state
```

but none of them has run through the model yet.

---

# Inspecting the Initial Scheduler State

We can print only the request IDs instead of printing every tensor inside the dictionaries:

```python
print("waiting: ", end="")

for i in waiting:
    print(i["id"], "\t", end="")

print("\nrunning: ", end="")

for j in running:
    print(j["id"], "\t", end="")
```

Here:

```python
for i in waiting:
```

iterates through each request dictionary.

Then:

```python
i["id"]
```

extracts only:

```text
A
B
C
```

rather than printing the entire request.

Initially we get:

```text
waiting: A    B    C
running:
```

---

# Scheduling Requests

We continue using the scheduler from before:

```python
def schedule():

    while waiting and len(running) < max_running:

        request = waiting.pop(0)

        running.append(request)
```

Because:

```python
max_running = 2
```

the first scheduling pass moves:

```text
A → running
B → running
```

while C remains waiting:

```text
waiting = [C]
running = [A, B]
```

The scheduler itself does not need to understand tensors, KV caches, or PyTorch.

It only manages request state.

---

# Running Real Model Inference

Our fake:

```python
request["generated_tokens"] += 1
```

is replaced with a real model forward pass.

```python
@torch.inference_mode()
def execute_iteration():

    for request in running:

        outputs = model(
            input_ids=request["input_ids"],
            attention_mask=request["attention_mask"],
            past_key_values=request["cache"],
            use_cache=True,
        )

        request["cache"] = outputs.past_key_values

        next_token = outputs.logits[:, -1, :].argmax(
            dim=-1,
            keepdim=True
        )

        request["generated_tokens"].append(
            next_token.item()
        )

        request["input_ids"] = next_token

        request["attention_mask"] = torch.cat(
            [
                request["attention_mask"],
                torch.ones(
                    (1, 1),
                    dtype=request["attention_mask"].dtype,
                    device=model.device,
                ),
            ],
            dim=-1,
        )
```

There is a lot happening here, so let's go through it carefully.

---

# `@torch.inference_mode()`

```python
@torch.inference_mode()
```

tells PyTorch that we are doing inference rather than training.

During training, PyTorch normally records operations so that gradients can later be calculated.

We do not need that for token generation.

So inference mode avoids unnecessary gradient-tracking work and memory usage.

---

# Looping Through Running Requests

```python
for request in running:
```

If:

```text
running = [A, B]
```

this currently means:

```text
run A
then
run B
```

This is important.

Our scheduler is continuously managing multiple active requests, but we have **not yet batched the actual GPU forward pass**.

Currently:

```text
A → model()
B → model()
```

Later we want:

```text
A ─┐
   ├── batched model()
B ─┘
```

This current intermediate step lets us understand the scheduler and request state before introducing the extra complexity of variable-length batched inference.

---

# Model Forward Pass

For each request:

```python
outputs = model(
    input_ids=request["input_ids"],
    attention_mask=request["attention_mask"],
    past_key_values=request["cache"],
    use_cache=True,
)
```

Let's follow request A.

On its first execution:

```text
input_ids
=
"The capital of France is"
```

and:

```text
cache
=
empty
```

So the model performs prefill:

```text
whole prompt
     ↓
transformer
     ↓
logits
+
KV cache
```

---

# `past_key_values`

We provide:

```python
past_key_values=request["cache"]
```

During the first iteration, the cache is empty.

During later iterations it contains the K/V states of all previous tokens.

That allows the model to avoid recomputing the complete sequence.

---

# `use_cache=True`

```python
use_cache=True
```

asks the model to return the updated KV cache after the forward pass.

The returned cache is available through:

```python
outputs.past_key_values
```

So we save it:

```python
request["cache"] = outputs.past_key_values
```

This means the request always holds its newest cache state.

---

# Understanding `outputs.logits`

The model returns:

```python
outputs.logits
```

with shape:

```text
[batch_size, sequence_length, vocabulary_size]
```

During A's initial prefill, suppose the prompt has five tokens.

Its logits could therefore have shape:

```text
[1, 5, vocab_size]
```

Why five sets of logits?

Because the transformer produces an output for every input position.

Conceptually:

```text
token 1 → logits
token 2 → logits
token 3 → logits
token 4 → logits
token 5 → logits
```

But generation only cares about:

> What token should come after the final current token?

So we select:

```python
outputs.logits[:, -1, :]
```

Let's break that indexing down.

Starting shape:

```text
[1, 5, vocab_size]
```

### First `:`

```python
:
```

means:

```text
take every batch element
```

### `-1`

```python
-1
```

means:

```text
take the final sequence position
```

### Final `:`

means:

```text
take every vocabulary logit
```

So:

```python
outputs.logits[:, -1, :]
```

changes:

```text
[1, 5, vocab_size]
```

into:

```text
[1, vocab_size]
```

Now we have one score for every possible next token.

---

# Greedy Token Selection

We then use:

```python
.argmax(
    dim=-1,
    keepdim=True
)
```

`argmax` returns the index containing the largest value.

Since each vocabulary position corresponds to a token ID, this means:

```text
choose the token with the largest logit
```

This is greedy decoding.

Suppose:

```text
" Paris"
```

has the largest score.

Then:

```python
next_token
```

might look like:

```text
tensor([[12095]], device="cuda:0")
```

Its shape is:

```text
[1, 1]
```

We keep that second dimension because the model expects token sequences to have:

```text
[batch_size, sequence_length]
```

Even though the new sequence length is just `1`.

---

# Saving the Generated Token

We store:

```python
request["generated_tokens"].append(
    next_token.item()
)
```

`next_token` is still a tensor:

```text
tensor([[12095]])
```

Calling:

```python
.item()
```

extracts the scalar Python integer:

```text
12095
```

So:

```python
generated_tokens
```

becomes:

```text
[12095]
```

---

# Switching From Prefill to Decode

This line is especially important:

```python
request["input_ids"] = next_token
```

During the first iteration:

```text
input_ids
=
whole prompt
```

After that iteration, the prompt has already been stored in the KV cache.

We do **not** want to send:

```text
The capital of France is Paris
```

again from scratch.

Instead, the next model call receives only:

```text
Paris
```

while the cache already contains:

```text
The capital of France is
```

So generation becomes:

```text
FIRST ITERATION

full prompt
    ↓
PREFILL
    ↓
cache(prompt)
    ↓
token 1
```

Then:

```text
SECOND ITERATION

token 1
+
cache(prompt)
    ↓
DECODE
    ↓
token 2
```

Then:

```text
THIRD ITERATION

token 2
+
cache(prompt + token 1)
    ↓
DECODE
    ↓
token 3
```

This is the same KV-cache mechanism we learned earlier, but now every request owns its own cache inside the scheduler.

---

# Extending the Attention Mask

After generating a token, our total context becomes one token longer.

Suppose before generation:

```text
context length = 5
```

The mask is:

```text
[1, 1, 1, 1, 1]
```

After generating one token:

```text
context length = 6
```

so we need:

```text
[1, 1, 1, 1, 1, 1]
```

We create one extra `1`:

```python
torch.ones(
    (1, 1),
    dtype=request["attention_mask"].dtype,
    device=model.device,
)
```

Its shape is:

```text
[1, 1]
```

Then concatenate it:

```python
torch.cat(
    [
        request["attention_mask"],
        new_one
    ],
    dim=-1
)
```

Suppose the original mask has shape:

```text
[1, 5]
```

and our new tensor has:

```text
[1, 1]
```

Concatenating along:

```python
dim=-1
```

means concatenate along the last dimension, which here is the sequence dimension.

So:

```text
[1, 5] + [1, 1]
```

becomes:

```text
[1, 6]
```

The actual mask becomes:

```text
[1, 1, 1, 1, 1, 1]
```

On the next iteration:

```text
[1, 7]
```

then:

```text
[1, 8]
```

and so on.

---

# Checking Whether a Request Finished

After every execution iteration, we inspect every running request.

```python
def check_finished():

    for request in running:

        reached_max_tokens = (
            len(request["generated_tokens"])
            >= request["max_new_tokens"]
        )

        generated_eos = (
            len(request["generated_tokens"]) > 0
            and
            request["generated_tokens"][-1]
            == tokenizer.eos_token_id
        )

        if reached_max_tokens or generated_eos:
            request["finished"] = True
```

There are two stopping conditions.

---

## Maximum Generated Tokens

```python
reached_max_tokens = (
    len(request["generated_tokens"])
    >= request["max_new_tokens"]
)
```

Suppose A has:

```python
max_new_tokens = 5
```

and:

```python
generated_tokens = [
    12095,
    13,
    1084,
    374,
    279
]
```

Then:

```python
len(request["generated_tokens"])
```

returns:

```text
5
```

so:

```text
5 >= 5
```

is:

```text
True
```

and A finishes.

---

# EOS

Models also have a special token representing:

```text
End Of Sequence
```

available as:

```python
tokenizer.eos_token_id
```

We inspect the final generated token:

```python
request["generated_tokens"][-1]
```

`-1` means:

```text
last element of the list
```

For example:

```python
tokens = [10, 20, 30]
tokens[-1]
```

returns:

```text
30
```

So:

```python
request["generated_tokens"][-1] == tokenizer.eos_token_id
```

checks whether the latest generated token was EOS.

We first check:

```python
len(request["generated_tokens"]) > 0
```

because trying:

```python
[][-1]
```

would cause an error.

---

# `and` Short-Circuiting

This expression:

```python
len(request["generated_tokens"]) > 0
and
request["generated_tokens"][-1] == tokenizer.eos_token_id
```

uses Python's short-circuit behavior.

If:

```python
len(...) > 0
```

is already `False`, Python does not evaluate the second part.

That prevents us from trying to access:

```python
[-1]
```

on an empty list.

---

# Either Condition Can Finish the Request

Finally:

```python
if reached_max_tokens or generated_eos:
    request["finished"] = True
```

The request ends if either:

```text
max_new_tokens reached
```

or:

```text
EOS generated
```

---

# Removing Finished Requests

Once a request is marked finished:

```python
def remove_finished():

    for request in running[:]:

        if request["finished"]:
            running.remove(request)
            finished.append(request)
```

We use:

```python
running[:]
```

instead of:

```python
running
```

because:

```python
running[:]
```

creates a shallow copy of the list.

This matters because we are modifying the original list while iterating.

If:

```text
running = [A, B]
```

and A finishes:

```python
running.remove(A)
```

changes the original list immediately.

Looping over a copy prevents those changes from interfering with the iteration.

After A finishes:

```text
running  = [B]
finished = [A]
```

Its execution slot is now free.

---

# The Continuous Engine Loop

Now everything comes together:

```python
step = 0

while waiting or running:

    step += 1

    # Fill available execution slots
    schedule()

    print(
        f"Step {step} | Running:",
        [request["id"] for request in running]
    )

    # Actual model inference
    execute_iteration()

    # Check EOS / max token limit
    check_finished()

    # Free completed request slots
    remove_finished()
```

This loop is the most important part of the section.

---

# Understanding `while waiting or running`

The loop continues while either:

```text
waiting contains requests
```

or:

```text
running contains requests
```

For example:

```text
waiting = [C]
running = [A, B]
```

means:

```python
waiting or running
```

evaluates to true.

Even later:

```text
waiting = []
running = [C]
```

the loop still continues because C is still running.

Only when:

```text
waiting = []
running = []
```

does the loop terminate.

---

# Why `schedule()` Is Inside the Loop

This is where the continuous scheduling behavior comes from.

The order is:

```text
schedule
   ↓
execute one generation iteration
   ↓
check completed requests
   ↓
remove completed requests
   ↓
go back to schedule
```

The scheduler therefore gets another opportunity after every generation iteration.

That allows requests to enter and leave the active set.

---

# Understanding Our Actual Output

Our engine produced:

```text
Step 1 | Running: ['A', 'B']
Step 2 | Running: ['A', 'B']
Step 3 | Running: ['A', 'B']
Step 4 | Running: ['A', 'B']
Step 5 | Running: ['A', 'B']

Step 6 | Running: ['B', 'C']

Step 7 | Running: ['B', 'C']
Step 8 | Running: ['B', 'C']

Step 9 | Running: ['C']
Step 10 | Running: ['C']
Step 11 | Running: ['C']
```

Let's understand exactly what happened.

Initially:

```text
waiting = [A, B, C]
running = []
```

Our capacity is:

```python
max_running = 2
```

So during Step 1:

```python
schedule()
```

moves A and B:

```text
running = [A, B]
waiting = [C]
```

---

## Steps 1–5

We see:

```text
[A, B]
[A, B]
[A, B]
[A, B]
[A, B]
```

A has:

```python
max_new_tokens = 5
```

Therefore after its fifth inference iteration:

```text
A generated 5 tokens
```

and:

```python
check_finished()
```

marks A finished.

Then:

```python
remove_finished()
```

changes:

```text
running = [A, B]
```

into:

```text
running = [B]
```

C is still waiting.

---

# Step 6: The Important Continuous Scheduling Moment

The outer `while` loop starts again.

The first thing it does is:

```python
schedule()
```

Now:

```text
running = [B]
max_running = 2
```

so one slot is available.

C moves from:

```text
waiting
```

to:

```text
running
```

giving:

```text
running = [B, C]
```

That is why the output becomes:

```text
Step 6 | Running: ['B', 'C']
```

This transition:

```text
[A, B]
   ↓
[B, C]
```

is the main continuous scheduling behavior we wanted.

We did not wait for B to finish before starting C.

---

# Why B Disappears After Step 8

B has:

```python
max_new_tokens = 8
```

It ran during:

```text
Steps 1–8
```

so after Step 8:

```text
B generated 8 tokens
```

and is moved into:

```text
finished
```

Now only C remains.

---

# Why C Runs Until Step 11

C entered during Step 6.

It has:

```python
max_new_tokens = 6
```

Therefore its six iterations are:

```text
Step 6
Step 7
Step 8
Step 9
Step 10
Step 11
```

After Step 11 it reaches six generated tokens and finishes.

So the final state becomes:

```text
waiting  = []
running  = []
finished = [A, B, C]
```

and the outer `while` loop stops.

---

# Decoding the Generated Tokens

We stored token IDs rather than text.

So:

```python
for request in finished:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True
    )

    print(f"\nRequest {request['id']}:")
    print(text)
```

converts each request's generated IDs back into readable text.

Our actual outputs were:

```text
Request A:
 Paris. It is the
```

```text
Request B:
 ____
A. Venus
B.
```

```text
Request C:
 a field of artificial intelligence that
```

These outputs are short because we intentionally used small:

```python
max_new_tokens
```

values.

The point here is not generation quality.

The important result is that real model inference now follows our scheduler.

---

# What We Have Built So Far

Each request now moves through:

```text
                        ┌─────────────┐
                        │   waiting   │
                        └──────┬──────┘
                               │
                           schedule()
                               │
                               ▼
                        ┌─────────────┐
                        │   running   │
                        └──────┬──────┘
                               │
                               ▼
                         model forward
                               │
                      ┌────────┴────────┐
                      │                 │
                  first step       later steps
                      │                 │
                      ▼                 ▼
                   PREFILL           DECODE
                      │            using KV cache
                      └────────┬────────┘
                               │
                          next token
                               │
                               ▼
                   EOS / max_new_tokens?
                         │          │
                        no         yes
                         │          │
                         ▼          ▼
                      running    finished
```

And because our outer engine loop repeatedly calls:

```python
schedule()
```

the set of running requests can change between iterations.

---

# An Important Limitation

We should be precise about what exists at this point.

We have:

```text
real model inference               ✅
real KV caching                    ✅
prefill → decode transition        ✅
multiple request lifecycle         ✅
continuous scheduling              ✅
request admission between steps    ✅
EOS / token-limit stopping         ✅
```

But we do **not yet have true batched GPU execution**.

This line:

```python
for request in running:
```

means that if:

```text
running = [A, B]
```

we execute:

```text
model(A)
model(B)
```

as two independent forward passes.

The GPU is not yet receiving:

```text
[A, B]
```

as one batch.

So our engine currently looks like:

```text
                 Scheduler
                     │
             running = [A, B]
                  /       \
                 /         \
             model(A)    model(B)
```

The next step is to move toward:

```text
                 Scheduler
                     │
             running = [A, B]
                     │
                     ▼
              build one batch
                     │
                     ▼
                 model(...)
                     │
             one GPU forward
```

That introduces the next major problem:

> **A and B do not necessarily have the same sequence length.**

For example:

```text
A prompt = 5 tokens
B prompt = 9 tokens
```

and after continuous scheduling we may eventually have:

```text
B = already decoding
C = brand-new request requiring prefill
```

So before we can perform real batched execution, we need to understand how multiple sequences with different lengths can be represented together and how **prefill and decode work interact inside a continuously changing batch**.


In [47]:
#batching in inference

waiting = []
running = []
finished = []

max_running = 2

waiting.append(
    create_request(
        "A",
        "The capital of France is",
        max_new_tokens=5
    )
)

waiting.append(
    create_request(
        "B",
        "The largest planet in our solar system is",
        max_new_tokens=8
    )
)

waiting.append(
    create_request(
        "C",
        "Machine learning is",
        max_new_tokens=6
    )
)

schedule()

In [48]:
print([request["id"] for request in running])
print([request["id"] for request in waiting])

['A', 'B']
['C']


In [51]:
#different shapes = hard to batch it togethere
for request in running:
    print(
        request["id"],
        request["input_ids"].shape
    )

A torch.Size([1, 5])
B torch.Size([1, 8])


In [50]:
#using left padding

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

In [52]:
#the main batching logic

batch = tokenizer.pad(
    {
        "input_ids": [
            request["input_ids"].squeeze(0)
            for request in running
        ],

        "attention_mask": [
            request["attention_mask"].squeeze(0)
            for request in running
        ],
    },
    padding=True,
    return_tensors="pt",
).to(model.device)

In [53]:
# inspecting them after padding and batching
print("input_ids shape:")
print(batch["input_ids"].shape)

print("\nattention_mask:")
print(batch["attention_mask"])

input_ids shape:
torch.Size([2, 8])

attention_mask:
tensor([[0, 0, 0, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')


In [54]:
#performing one model fwd pass

batch_cache = DynamicCache(config=model.config)
with torch.inference_mode():

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        past_key_values=batch_cache,
        use_cache=True,
    )





In [55]:
# shape of output batch after 1 fwd pass
print(outputs.logits.shape)

torch.Size([2, 8, 151936])


In [56]:
#doing next decode
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True
)

print(next_tokens)
print(next_tokens.shape)

tensor([[12095],
        [ 1304]], device='cuda:0')
torch.Size([2, 1])


In [57]:
#decoding them seperately:

for i, request in enumerate(running):

    token_id = next_tokens[i].item()

    print(
        request["id"],
        "→",
        tokenizer.decode([token_id])
    )

A →  Paris
B →  __


## Batching Multiple Requests Into One Model Forward Pass

So far, our scheduler can keep multiple requests in:

```text
waiting
running
finished
```

and continuously admit new requests when execution slots become free.

However, our previous `execute_iteration()` still did this:

```python
for request in running:
    outputs = model(...)
```

So if:

```text
running = [A, B]
```

the actual GPU execution was:

```text
model(A)
model(B)
```

These are two separate forward passes.

The next step is to turn the currently running requests into a **real model batch**:

```text
A ─┐
   ├── model([A, B])
B ─┘
```

so that both requests are processed in the same forward pass.

---

## Creating Fresh Requests

First, reset the request queues:

```python
waiting = []
running = []
finished = []

max_running = 2
```

Create three requests:

```python
waiting.append(
    create_request(
        "A",
        "The capital of France is",
        max_new_tokens=5
    )
)

waiting.append(
    create_request(
        "B",
        "The largest planet in our solar system is",
        max_new_tokens=8
    )
)

waiting.append(
    create_request(
        "C",
        "Machine learning is",
        max_new_tokens=6
    )
)
```

Now run the scheduler:

```python
schedule()
```

Inspect the state:

```python
print([request["id"] for request in running])
print([request["id"] for request in waiting])
```

Output:

```text
['A', 'B']
['C']
```

Because:

```python
max_running = 2
```

only A and B are admitted.

So:

```text
running = [A, B]
waiting = [C]
```

---

# The First Batching Problem: Different Sequence Lengths

Let's inspect the tokenized prompt shapes:

```python
for request in running:

    print(
        request["id"],
        request["input_ids"].shape
    )
```

Output:

```text
A torch.Size([1, 5])
B torch.Size([1, 8])
```

Request A contains 5 prompt tokens.

Request B contains 8 prompt tokens.

So individually:

```text
A → [1, 5]

B → [1, 8]
```

The first dimension is the batch dimension.

Since each request was tokenized independently:

```text
batch_size = 1
```

The second dimension is the sequence length.

---

# Why Can't We Directly Batch Them?

A PyTorch tensor must be rectangular.

We cannot create something like:

```text
[
    [a, b, c, d, e],

    [a, b, c, d, e, f, g, h]
]
```

because one row has length `5` while the other has length `8`.

To combine them into a single tensor, they need the same sequence length.

We solve this using **padding**.

---

# Left Padding

We configure the tokenizer to pad shorter sequences on the left:

```python
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
```

Suppose A contains:

```text
[a, b, c, d, e]
```

and B contains:

```text
[b1, b2, b3, b4, b5, b6, b7, b8]
```

After left padding:

```text
A = [PAD, PAD, PAD, a,  b,  c,  d,  e]

B = [b1,  b2,  b3,  b4, b5, b6, b7, b8]
```

Both sequences now have length:

```text
8
```

This allows them to become one tensor with shape:

```text
[2, 8]
```

where:

```text
2 = batch size
8 = padded sequence length
```

---

# Why Left Padding?

Later we obtain next-token logits using:

```python
outputs.logits[:, -1, :]
```

The `-1` means:

```text
take the final sequence position
```

With left padding:

```text
[PAD PAD PAD actual actual actual actual LAST]
```

the final position is still the last actual prompt token.

That makes:

```python
outputs.logits[:, -1, :]
```

a convenient way to obtain one next-token prediction for every request.

---

# Building the Batch

Now we combine the currently running requests:

```python
batch = tokenizer.pad(
    {
        "input_ids": [
            request["input_ids"].squeeze(0)
            for request in running
        ],

        "attention_mask": [
            request["attention_mask"].squeeze(0)
            for request in running
        ],
    },
    padding=True,
    return_tensors="pt",
).to(model.device)
```

There are a few things happening here.

---

## Collecting `input_ids`

This:

```python
[
    request["input_ids"].squeeze(0)
    for request in running
]
```

is a Python list comprehension.

It is equivalent to doing:

```python
input_sequences = []

for request in running:
    input_sequences.append(
        request["input_ids"].squeeze(0)
    )
```

If:

```text
running = [A, B]
```

we collect the token sequences for:

```text
A
B
```

into one Python list.

---

# Why `.squeeze(0)`?

A's `input_ids` currently has shape:

```text
[1, 5]
```

B's has:

```text
[1, 8]
```

The leading `1` exists because each request was originally tokenized as its own batch.

But `tokenizer.pad()` wants a collection of individual sequences.

So we use:

```python
.squeeze(0)
```

For A:

```text
before:
[1, 5]

after:
[5]
```

For B:

```text
before:
[1, 8]

after:
[8]
```

Now `tokenizer.pad()` receives something conceptually like:

```text
[
    sequence_A,
    sequence_B
]
```

rather than:

```text
[
    batch_containing_A,
    batch_containing_B
]
```

---

# Padding the Attention Masks

We do the same thing for:

```python
request["attention_mask"]
```

using:

```python
[
    request["attention_mask"].squeeze(0)
    for request in running
]
```

Before padding, A's mask might be:

```text
[1, 1, 1, 1, 1]
```

while B's is:

```text
[1, 1, 1, 1, 1, 1, 1, 1]
```

After left padding, A becomes:

```text
[0, 0, 0, 1, 1, 1, 1, 1]
```

while B remains:

```text
[1, 1, 1, 1, 1, 1, 1, 1]
```

The zeros correspond to padding positions.

The ones correspond to actual tokens.

---

# Inspecting the Batched Inputs

We can inspect the result:

```python
print("input_ids shape:")
print(batch["input_ids"].shape)

print("\nattention_mask:")
print(batch["attention_mask"])
```

Output:

```text
input_ids shape:
torch.Size([2, 8])

attention_mask:
tensor([
    [0, 0, 0, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1]
], device='cuda:0')
```

This is our first real batch.

Previously we had:

```text
A → [1, 5]
B → [1, 8]
```

Now we have:

```text
batch → [2, 8]
```

The model can therefore process A and B together.

---

# Understanding the Attention Mask Again

The batch contains padding tokens for A:

```text
A:
[PAD PAD PAD token token token token token]
```

Without an attention mask, those padding positions could incorrectly participate in attention.

The mask:

```text
[0, 0, 0, 1, 1, 1, 1, 1]
```

means:

```text
0 → ignore this padded position

1 → this is part of the actual sequence
```

For B there was no padding:

```text
[1, 1, 1, 1, 1, 1, 1, 1]
```

because B already had the maximum sequence length in this batch.

This is one of the reasons attention masks become much more important once we begin batching different sequence lengths.

---

# Creating a Batched KV Cache

Previously, every request had its own:

```python
DynamicCache(...)
```

For this experiment, A and B are going through the model together.

So we create one cache for the entire batch:

```python
batch_cache = DynamicCache(
    config=model.config
)
```

Initially this cache is empty.

After the forward pass it will contain KV states corresponding to both rows of the batch.

Conceptually:

```text
batch dimension 0 → A
batch dimension 1 → B
```

---

# Performing One Batched Forward Pass

Now run:

```python
with torch.inference_mode():

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        past_key_values=batch_cache,
        use_cache=True,
    )
```

This is the major change.

Previously:

```text
model(A)

model(B)
```

Now:

```text
             ┌──────────────┐
A ──────────►│              │
             │    MODEL     │
B ──────────►│              │
             └──────────────┘
```

Both requests participate in the same model call.

---

# `torch.inference_mode()`

We use:

```python
with torch.inference_mode():
```

because we are generating tokens, not training the model.

During training, PyTorch normally records operations so gradients can later be calculated.

For inference we do not need this.

Inference mode avoids that unnecessary gradient tracking.

---

# Understanding the Batched Logits

Inspect:

```python
print(outputs.logits.shape)
```

Output:

```text
torch.Size([2, 8, 151936])
```

Remember that model logits have shape:

```text
[batch_size, sequence_length, vocabulary_size]
```

Therefore:

```text
2       → two requests: A and B

8       → padded sequence length

151936  → number of possible vocabulary tokens
```

This is important evidence that the model actually processed both requests together.

Before batching:

```text
A logits → [1, 5, 151936]

B logits → [1, 8, 151936]
```

Now:

```text
batched logits → [2, 8, 151936]
```

---

# Getting the Next Token for Each Request

Now we calculate:

```python
next_tokens = outputs.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True
)
```

Let's break this down.

The original shape is:

```text
[2, 8, 151936]
```

First:

```python
outputs.logits[:, -1, :]
```

means:

```text
:
take every request in the batch

-1
take the final sequence position

:
take every vocabulary logit
```

So the shape changes from:

```text
[2, 8, 151936]
```

to:

```text
[2, 151936]
```

We now have one vocabulary distribution for A and one for B.

Conceptually:

```text
row 0 → scores for A's next token

row 1 → scores for B's next token
```

---

# `argmax`

Then:

```python
.argmax(
    dim=-1,
    keepdim=True
)
```

finds the index with the highest logit for each row.

Since:

```text
dim=-1
```

means the final dimension, we perform `argmax` across the vocabulary.

The shape becomes:

```text
[2, 1]
```

Our actual result was:

```text
tensor([
    [12095],
    [ 1304]
], device='cuda:0')
```

with:

```text
torch.Size([2, 1])
```

This means:

```text
request A → token ID 12095

request B → token ID 1304
```

One batched forward produced one next token for **each active request**.

---

# Why `keepdim=True`?

Without:

```python
keepdim=True
```

the result would have shape:

```text
[2]
```

Instead we keep:

```text
[2, 1]
```

because token tensors passed into the model normally follow:

```text
[batch_size, sequence_length]
```

Here:

```text
batch_size = 2

sequence_length = 1
```

This shape will become useful when we perform batched decode later.

---

# Decoding Each Result Separately

Even though the model processed A and B together, the generated tokens still belong to separate requests.

We decode them using:

```python
for i, request in enumerate(running):

    token_id = next_tokens[i].item()

    print(
        request["id"],
        "→",
        tokenizer.decode([token_id])
    )
```

Our output was:

```text
A →  Paris
B →  __
```

---

# Understanding `enumerate`

Suppose:

```text
running = [A, B]
```

Then:

```python
enumerate(running)
```

produces:

```text
i = 0, request = A

i = 1, request = B
```

The `next_tokens` tensor has the exact same ordering:

```text
next_tokens[0] → output belonging to A

next_tokens[1] → output belonging to B
```

So:

```python
next_tokens[i]
```

lets us map each batched output back to its corresponding request.

This request-to-batch-position mapping is extremely important in inference engines.

---

# What We Have Achieved

Before:

```text
running = [A, B]

A → model() → token A

B → model() → token B
```

Two forward passes.

Now:

```text
running = [A, B]

      ┌───────────────┐
A ───►│               │
      │     MODEL     │───► token A
B ───►│               │───► token B
      └───────────────┘
```

One forward pass.

So we now have:

```text
multiple active requests        ✅
different prompt lengths        ✅
left padding                    ✅
attention masking               ✅
one batched input tensor        ✅
one model forward               ✅
one next token per request      ✅
```

This is our first actual **batched GPU execution**.

---

# The Next Problem

There is an important reason we have not yet inserted this directly into the continuous engine loop.

Right now, our batched cache corresponds to:

```text
[A, B]
```

Suppose after several decode steps A finishes.

The scheduler changes:

```text
[A, B]
   ↓
[B]
```

and then admits C:

```text
[B]
 ↓
[B, C]
```

The Python scheduler can easily change the active request list.

But the existing KV cache was created with batch positions:

```text
row 0 → A
row 1 → B
```

Now our active requests are:

```text
row ? → B
row ? → C
```

C also has no KV cache yet because it has not performed prefill, while B may already have several cached decode tokens.

So simply changing:

```text
running = [A, B]
```

to:

```text
running = [B, C]
```

does not automatically make the batched KV cache correct.

This is the next major issue we need to understand: but we will end the notebook now as we will cover the B part of continuous batching in an another notebook

